# LlmEdith : pré-entraînement JAX sur TPU (Kaggle)
**Réglages Kaggle** : *Accelerator* = **TPU VM v5e-8** (ou v3-8), *Internet* = **ON**.
*Add-ons → Secrets* : `HF_TOKEN` (jeton HF en écriture), `GITHUB_TOKEN` seulement si ton dépôt est privé.

Chaque session reprend automatiquement au dernier checkpoint (sauvegardé toutes les 30 min sur le Hub),
puis s'arrête proprement avant la limite de 9 h. Il suffit de relancer le notebook chaque semaine.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
try:
    GH = s.get_secret('GITHUB_TOKEN')  # seulement si le dépôt GitHub est privé
except Exception:
    GH = None
os.environ['LLMEDITH_HUB_REPO'] = 'TON_PSEUDO_HF/llmedith-checkpoints'
repo = 'https://github.com/HarpeLm/LlmEdith.git'.replace('https://', f'https://{GH}@') if GH else 'https://github.com/HarpeLm/LlmEdith.git'
if not os.path.exists('LlmEdith'):
    subprocess.run(['git', 'clone', '-q', repo, 'LlmEdith'], check=True)
else:
    subprocess.run(['git', '-C', 'LlmEdith', 'pull', '-q'], check=True)
os.chdir('LlmEdith')

In [ ]:
!pip install -q -e . optax
try:
    import jax
except ImportError:
    !pip install -q "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
import jax
print(jax.devices())

In [ ]:
!python -m llmedith.jax_impl.train --config configs/main.yaml --resume --max_minutes 500 \
    --override data.hub_repo=TON_PSEUDO_HF/llmedith-data

Courbe de loss de la session :

In [ ]:
import pandas as pd
log = pd.read_csv('runs/main/log.csv', names=['step', 'split', 'loss', 'lr', 'gnorm', 'tok_s'])
ax = log[log.split == 'train'].plot(x='step', y='loss', logy=True)
log[log.split == 'val'].plot(x='step', y='loss', ax=ax, style='o-', label='val');